# Music by Emotion — Inference

Classify the emotion of a music track using [`LaurenGurgiolo/Music_by_Emotion`](https://huggingface.co/LaurenGurgiolo/Music_by_Emotion), an Audio Spectrogram Transformer (AST) fine-tuned from `MIT/ast-finetuned-audioset-10-10-0.4593`.

The model expects mono audio resampled to **16 kHz** (it was trained on ~30-second clips).

**Inference tracks (compared below):**
- `My Drive/google_colab/Nero - Blame You.mp3`
- `My Drive/google_colab/The Cardigans - My Favourite Game (Ozma bootleg).mp3`

## 1. Install dependencies

In [13]:
!pip install -q transformers torch torchaudio librosa

## 2. Mount Google Drive

After mounting, `My Drive` is available at `/content/drive/MyDrive`.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

AUDIO_DIR = '/content/drive/MyDrive/google_colab'
AUDIO_PATHS = [
    f'{AUDIO_DIR}/Nero - Blame You.mp3',
    f'{AUDIO_DIR}/The Cardigans - My Favourite Game (Ozma bootleg).mp3',
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Load the model and processor

In [15]:
import torch
from transformers import AutoProcessor, AutoModelForAudioClassification

MODEL_ID = 'LaurenGurgiolo/Music_by_Emotion'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForAudioClassification.from_pretrained(MODEL_ID).to(device).eval()

# The published model config only carries generic LABEL_0..LABEL_10 names.
# These are the real emotion classes, taken from the training dataset's
# ClassLabel feature order (LaurenGurgiolo/Music_by_Emotion), in index order.
EMOTION_LABELS = [
    'Angry',      # 0
    'Contempt',   # 1
    'Disgust',    # 2
    'Fear',       # 3
    'Happy',      # 4
    'Neutral',    # 5
    'Sad',        # 6
    'Sleepy',     # 7
    'Surprised',  # 8
    'Bad',        # 9
    'Boring',     # 10
]

# Apply the real names so downstream code can read them off the model config.
model.config.id2label = {i: name for i, name in enumerate(EMOTION_LABELS)}
model.config.label2id = {name: i for i, name in enumerate(EMOTION_LABELS)}

# Sampling rate the processor expects (16 kHz for AST)
TARGET_SR = processor.sampling_rate
print('Target sampling rate:', TARGET_SR)
print('Emotion labels:', model.config.id2label)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Target sampling rate: 16000
Emotion labels: {0: 'Angry', 1: 'Contempt', 2: 'Disgust', 3: 'Fear', 4: 'Happy', 5: 'Neutral', 6: 'Sad', 7: 'Sleepy', 8: 'Surprised', 9: 'Bad', 10: 'Boring'}


## 4. Inference helper

Load a track (mono, resampled to the model's target rate), run the model, and
return the per-emotion probabilities.

In [16]:
import os
import librosa


def classify(path):
    """Return (probs_tensor, duration_seconds) for one audio file."""
    waveform, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    inputs = processor(waveform, sampling_rate=TARGET_SR, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu()
    return probs, len(waveform) / sr


# Run inference on every track up front.
results = {}
for path in AUDIO_PATHS:
    name = os.path.splitext(os.path.basename(path))[0]
    probs, duration = classify(path)
    results[name] = probs
    print(f'{name} — {duration:.1f}s')

Nero - Blame You — 200.5s
The Cardigans - My Favourite Game (Ozma bootleg) — 270.2s


## 5. Top prediction per track

In [17]:
for name, probs in results.items():
    top_id = int(probs.argmax())
    print(f'{name}')
    print(f'  Predicted emotion: {model.config.id2label[top_id]} '
          f'({probs[top_id].item():.2%})\n')

Nero - Blame You
  Predicted emotion: Sad (55.21%)

The Cardigans - My Favourite Game (Ozma bootleg)
  Predicted emotion: Happy (49.89%)



## 6. Side-by-side comparison

Full per-emotion probabilities for both tracks, sorted by the first track's scores.

In [18]:
import pandas as pd

comparison = pd.DataFrame(
    {name: [probs[i].item() for i in range(len(EMOTION_LABELS))]
     for name, probs in results.items()},
    index=EMOTION_LABELS,
)
comparison.index.name = 'emotion'

# Sort by the first track's probability for easy reading.
comparison = comparison.sort_values(by=comparison.columns[0], ascending=False)

# Show as percentages.
(comparison * 100).round(2).style.format('{:.2f}%')

,Nero - Blame You,The Cardigans - My Favourite Game (Ozma bootleg)
emotion,,
Sad,55.21%,0.11%
Boring,13.70%,28.76%
Contempt,11.57%,7.16%
Disgust,10.59%,2.09%
Surprised,3.73%,0.89%
Sleepy,2.86%,1.74%
Fear,0.99%,4.42%
Angry,0.72%,0.23%
Bad,0.50%,3.51%
